# 02 - Verificacion de Eventos en Kafka

Este notebook verifica que los eventos del producer esten llegando correctamente a Kafka.

## 1. Instalacion de kafka-python

In [1]:
import sys
!{sys.executable} -m pip install kafka-python-ng pandas python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 4.4 MB/s eta 0:00:00a 0:00:01


## 2. Consumo de mensajes de Kafka

In [2]:
from kafka import KafkaConsumer
import json
import time

KAFKA_BROKER = "kafka:29092"
TOPIC = "iot.air_quality.streaming"
MAX_MENSAJES = 20
TIEMPO_ESPERA = 30

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=KAFKA_BROKER,
    auto_offset_reset="latest",
    value_deserializer=lambda m: json.loads(m.decode("utf-8")),
    consumer_timeout_ms=1000
)

print(f"Consumiendo mensajes del topico: {TOPIC}")
print(f"Esperando hasta {TIEMPO_ESPERA}s o {MAX_MENSAJES} mensajes...\n")

messages = []
start = time.time()
try:
    while time.time() - start < TIEMPO_ESPERA and len(messages) < MAX_MENSAJES:
        msg_pack = consumer.poll(timeout_ms=1000)
        for tp, records in msg_pack.items():
            for record in records:
                messages.append(record.value)
                print(f"[{len(messages)}] Estacion: {record.value.get('estacion')} | Temp: {record.value.get('temperatura')} | IAQ: {record.value.get('iaq')}")
except KeyboardInterrupt:
    print("\n[Consumo interrumpido por el usuario]")

print(f"\nTotal mensajes recibidos: {len(messages)}")
consumer.close()

Consumiendo mensajes del topico: iot.air_quality.streaming
Esperando hasta 30s o 20 mensajes...

[1] Estacion: ESP32_04 | Temp: 12.1 | IAQ: 161.4507
[2] Estacion: ESP32_03 | Temp: 12.8 | IAQ: 137.4507
[3] Estacion: ESP32_01 | Temp: 12.18 | IAQ: 141.4507
[4] Estacion: ESP32_02 | Temp: 12.0 | IAQ: 137.4507
[5] Estacion: ESP32_05 | Temp: 12.18 | IAQ: 140.4507
[6] Estacion: ESP32_02 | Temp: 11.85 | IAQ: 141.4507
[7] Estacion: ESP32_03 | Temp: 12.73 | IAQ: 140.4507
[8] Estacion: ESP32_01 | Temp: 12.18 | IAQ: 135.4507
[9] Estacion: ESP32_04 | Temp: 12.09 | IAQ: 155.4507
[10] Estacion: ESP32_05 | Temp: 12.14 | IAQ: 138.4507
[11] Estacion: ESP32_04 | Temp: 12.15 | IAQ: 154.4507
[12] Estacion: ESP32_05 | Temp: 12.24 | IAQ: 132.4507
[13] Estacion: ESP32_01 | Temp: 12.18 | IAQ: 139.4507
[14] Estacion: ESP32_02 | Temp: 12.09 | IAQ: 132.4507
[15] Estacion: ESP32_03 | Temp: 12.27 | IAQ: 139.4507

Total mensajes recibidos: 15


## 3. Analisis de mensajes recibidos

In [3]:
import pandas as pd

if messages:
    df = pd.DataFrame(messages)
    print("\nEstructura del evento:")
    print(df.columns.tolist())
    print("\nPrimer evento completo:")
    print(df.iloc[0].to_dict())
    print("\nEstaciones detectadas:")
    print(df["estacion"].value_counts())
else:
    print("No se recibieron mensajes. Verifica que el producer este corriendo.")


Estructura del evento:
['estacion', 'temperatura', 'humedad', 'presion', 'altura', 'gas', 'iaq', 'eco2', 'VOC', 'calidad_aire', 'created_at', 'event_timestamp', 'delayed']

Primer evento completo:
{'estacion': 'ESP32_04', 'temperatura': 12.1, 'humedad': 33.91, 'presion': 648.24, 'altura': 3598.33, 'gas': 49372, 'iaq': 161.4507, 'eco2': 1688.546, 'VOC': 0.43, 'calidad_aire': 'MALA', 'created_at': '2026-06-15T16:46:52.830963', 'event_timestamp': 1781542012830, 'delayed': nan}

Estaciones detectadas:
estacion
ESP32_04    3
ESP32_03    3
ESP32_01    3
ESP32_02    3
ESP32_05    3
Name: count, dtype: int64


## 4. Verificacion de eventos retrasados (ESP32_05)

In [4]:
if messages:
    delayed = [m for m in messages if m.get("delayed", False)]
    print(f"Eventos retrasados detectados: {len(delayed)}")
    if delayed:
        print("\nEjemplo de evento retrasado:")
        print(f"  created_at: {delayed[0].get('created_at')}")
        print(f"  estacion: {delayed[0].get('estacion')}")
        print(f"  delayed: {delayed[0].get('delayed')}")

Eventos retrasados detectados: 3

Ejemplo de evento retrasado:
  created_at: 2026-06-15T16:46:47.831014
  estacion: ESP32_05
  delayed: True


## 5. Verificacion de latencia

In [5]:
from datetime import datetime

if messages:
    latencias = []
    now = datetime.utcnow().timestamp() * 1000
    for m in messages:
        if m.get("event_timestamp"):
            lat = now - m["event_timestamp"]
            latencias.append(lat)
    
    if latencias:
        print(f"Latencia promedio: {sum(latencias)/len(latencias):.0f} ms")
        print(f"Latencia minima: {min(latencias):.0f} ms")
        print(f"Latencia maxima: {max(latencias):.0f} ms")

Latencia promedio: 17371 ms
Latencia minima: 6368 ms
Latencia maxima: 31373 ms
